In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ocha_stratus as stratus
import pandas as pd
from dotenv import load_dotenv
from src.constants import STATE_CONFIG
from src.datasources import glofas, grrr

load_dotenv()

ANALYSIS_START_YEAR = 2003
ANALYSIS_END_YEAR = 2024
WET_MONTHS = [8, 9, 10, 11, 12]
RP_LEVELS = [2, 5]

In [ ]:
STATE = "Adamawa"  # or "Benue"
cfg = STATE_CONFIG[STATE]

FIGURES_DIR = "figures"

## Load data

In [ ]:
# GloFAS reanalysis (truth)
df_gf_ra = glofas.load_glofas_reanalysis(cfg["glofas_station"])
df_gf_ra["time"] = pd.to_datetime(df_gf_ra["time"])

# GloFAS reforecast (ensemble)
df_gf_rf = glofas.load_glofas_reforecast(cfg["glofas_station"], product_type="ensemble")
df_gf_rf["time"] = pd.to_datetime(df_gf_rf["time"])
df_gf_rf["valid_time"] = pd.to_datetime(df_gf_rf["valid_time"])

# Google GRRR reanalysis (truth)
ds_grrr_ra = grrr.load_reanalysis(gauge=cfg["google_gauge"])
df_grrr_ra = grrr.process_reanalysis(ds_grrr_ra)
df_grrr_ra["valid_time"] = pd.to_datetime(df_grrr_ra["valid_time"])

# Google GRRR reforecast
ds_grrr_rf = grrr.load_reforecast(gauge=cfg["google_gauge"])
df_grrr_rf = grrr.process_reforecast(ds_grrr_rf)
df_grrr_rf["valid_time"] = pd.to_datetime(df_grrr_rf["valid_time"])
df_grrr_rf["issue_time"] = pd.to_datetime(df_grrr_rf["issue_time"])

# Google return period thresholds
ds_rp = grrr.load_return_periods(gauge=cfg["google_gauge"])
print(ds_rp)

## Return period thresholds

GloFAS: derived from reanalysis annual maxima (wet-season).  
Google: from the official GCS return periods dataset.

In [ ]:
# GloFAS: annual max wet-season discharge from reanalysis
df_gf_ra_wet = df_gf_ra[df_gf_ra["time"].dt.month.isin(WET_MONTHS)]
gf_ra_annual = (
    df_gf_ra_wet.assign(year=df_gf_ra_wet["time"].dt.year)
    .groupby("year")["dis24"].max()
)
gf_thresholds = {rp: gf_ra_annual.quantile(1 - 1 / rp) for rp in RP_LEVELS}
print("GloFAS RP thresholds (m³/s):")
for rp, val in gf_thresholds.items():
    print(f"  {rp}-year: {val:,.0f}")

# Google: from official return periods zarr
df_rp = ds_rp.to_dataframe().reset_index()
print("\nGoogle return periods dataset:")
print(df_rp)

In [ ]:
# Inspect ds_rp structure and extract thresholds
# (adjust column/dim names below if needed after inspecting output above)
rp_col = [c for c in df_rp.columns if c not in ["gauge_id", "return_period"]][0]
google_thresholds = {
    rp: df_rp.loc[df_rp["return_period"] == rp, rp_col].iloc[0]
    for rp in RP_LEVELS
    if rp in df_rp["return_period"].values
}
print("Google RP thresholds (m³/s):")
for rp, val in google_thresholds.items():
    print(f"  {rp}-year: {val:,.0f}")

## Skill vs. leadtime

Spearman correlation and MAPE between daily reforecast and reanalysis values,
restricted to wet-season days (Aug–Dec) in the analysis period.

GloFAS: ensemble mean across members.  
Google: single deterministic reforecast.

In [ ]:
def skill_by_leadtime(df_rf, df_ra, rf_val_col, ra_val_col, ra_time_col="valid_time"):
    """Spearman r and MAPE between reforecast and reanalysis, grouped by leadtime."""
    # Filter reanalysis to wet season + analysis period
    ra_wet = df_ra[
        df_ra[ra_time_col].dt.month.isin(WET_MONTHS)
        & df_ra[ra_time_col].dt.year.between(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR)
    ].set_index(ra_time_col)[[ra_val_col]]

    records = []
    for lt, grp in df_rf.groupby("leadtime"):
        grp_wet = grp[
            grp["valid_time"].dt.month.isin(WET_MONTHS)
            & grp["valid_time"].dt.year.between(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR)
        ]
        merged = grp_wet.set_index("valid_time")[[rf_val_col]].join(
            ra_wet, how="inner"
        ).dropna()
        if len(merged) < 10:
            continue
        spearman = merged[rf_val_col].corr(merged[ra_val_col], method="spearman")
        pearson = merged[rf_val_col].corr(merged[ra_val_col])
        mape = (100 * (merged[rf_val_col] - merged[ra_val_col]).abs() / merged[ra_val_col].abs()).mean()
        records.append({"leadtime": lt, "spearman": spearman, "pearson": pearson, "mape": mape, "n": len(merged)})
    return pd.DataFrame(records).set_index("leadtime")


# GloFAS: ensemble mean per (valid_time, leadtime)
df_gf_rf_mean = (
    df_gf_rf.groupby(["valid_time", "leadtime"])["dis24"]
    .mean()
    .reset_index()
)

skill_gf = skill_by_leadtime(df_gf_rf_mean, df_gf_ra, "dis24", "dis24", ra_time_col="time")
skill_grrr = skill_by_leadtime(df_grrr_rf, df_grrr_ra, "streamflow", "streamflow")

print("GloFAS skill:")
print(skill_gf[["spearman", "pearson", "mape", "n"]].round(3))
print("\nGoogle GRRR skill:")
print(skill_grrr[["spearman", "pearson", "mape", "n"]].round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric, ylabel, ylim in [
    (axes[0], "spearman", "Spearman ρ (vs. reanalysis)", (0, 1)),
    (axes[1], "mape",     "MAPE (% vs. reanalysis)",     (0, None)),
]:
    if not skill_gf.empty:
        ax.plot(skill_gf.index, skill_gf[metric], marker="o", color="#007CE0", label="GloFAS")
    if not skill_grrr.empty:
        ax.plot(skill_grrr.index, skill_grrr[metric], marker="s", color="#1EBFB3", label="Google GRRR")
    ax.set_xlabel("Leadtime (days)")
    ax.set_ylabel(ylabel)
    ax.set_ylim(ylim)
    ax.legend(fontsize=9)
    ax.grid(axis="y", linewidth=0.5, alpha=0.5)

axes[0].set_title(f"Forecast correlation — {STATE}")
axes[1].set_title(f"Forecast MAPE — {STATE}")

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/{STATE.lower()}_skill_vs_leadtime.png", dpi=200, bbox_inches="tight")
plt.show()

## Annual exceedance detection by leadtime and RP

For each year and leadtime: was the wet-season annual max reforecast (ensemble mean)
above the RP threshold, and did the reanalysis annual max also exceed it?

Reports hit rate and false alarm rate vs. leadtime for each RP level.

In [ ]:
def annual_exceedance_detection(df_rf, df_ra, ra_thresholds, rf_val_col, ra_val_col,
                                ra_time_col="valid_time"):
    """Hit rate and false alarm rate vs. leadtime for each RP threshold.

    For each year and leadtime, compares the annual max ensemble-mean reforecast
    (wet-season) against the reanalysis annual max.
    """
    # Reanalysis annual max (wet season)
    ra_wet = df_ra[
        df_ra[ra_time_col].dt.month.isin(WET_MONTHS)
        & df_ra[ra_time_col].dt.year.between(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR)
    ]
    ra_annual = ra_wet.assign(year=ra_wet[ra_time_col].dt.year).groupby("year")[ra_val_col].max()

    # Reforecast annual max (wet season) per leadtime
    rf_wet = df_rf[
        df_rf["valid_time"].dt.month.isin(WET_MONTHS)
        & df_rf["valid_time"].dt.year.between(ANALYSIS_START_YEAR, ANALYSIS_END_YEAR)
    ].copy()
    rf_wet["year"] = rf_wet["valid_time"].dt.year
    rf_annual = rf_wet.groupby(["year", "leadtime"])[rf_val_col].max().reset_index()

    results = {}
    for rp, thresh in ra_thresholds.items():
        records = []
        for lt, grp in rf_annual.groupby("leadtime"):
            merged = grp.set_index("year")[[rf_val_col]].join(
                ra_annual.rename("ra"), how="inner"
            ).dropna()
            event = merged["ra"] > thresh
            trigger = merged[rf_val_col] > thresh
            tp = int((trigger & event).sum())
            fp = int((trigger & ~event).sum())
            fn = int((~trigger & event).sum())
            tn = int((~trigger & ~event).sum())
            hit_rate = tp / (tp + fn) if (tp + fn) > 0 else np.nan
            far = fp / (fp + tn) if (fp + tn) > 0 else np.nan
            records.append({"leadtime": lt, "hit_rate": hit_rate, "far": far,
                            "tp": tp, "fp": fp, "fn": fn, "tn": tn})
        results[rp] = pd.DataFrame(records).set_index("leadtime")
    return results


exc_gf = annual_exceedance_detection(
    df_gf_rf_mean, df_gf_ra, gf_thresholds, "dis24", "dis24", ra_time_col="time"
)
exc_grrr = annual_exceedance_detection(
    df_grrr_rf, df_grrr_ra, google_thresholds, "streamflow", "streamflow"
)

for rp in RP_LEVELS:
    print(f"\n--- {rp}-year RP ---")
    if rp in exc_gf:
        print("GloFAS:")
        print(exc_gf[rp][["hit_rate", "far", "tp", "fp", "fn", "tn"]].round(2))
    if rp in exc_grrr:
        print("Google GRRR:")
        print(exc_grrr[rp][["hit_rate", "far", "tp", "fp", "fn", "tn"]].round(2))

In [ ]:
fig, axes = plt.subplots(len(RP_LEVELS), 2, figsize=(12, 4 * len(RP_LEVELS)))
if len(RP_LEVELS) == 1:
    axes = axes[np.newaxis, :]

for row, rp in enumerate(RP_LEVELS):
    for col, (metric, ylabel) in enumerate([
        ("hit_rate", "Hit rate (TP / (TP+FN))"),
        ("far",      "False alarm rate (FP / (FP+TN))"),
    ]):
        ax = axes[row, col]
        if rp in exc_gf and not exc_gf[rp].empty:
            ax.plot(exc_gf[rp].index, exc_gf[rp][metric],
                    marker="o", color="#007CE0", label="GloFAS")
        if rp in exc_grrr and not exc_grrr[rp].empty:
            ax.plot(exc_grrr[rp].index, exc_grrr[rp][metric],
                    marker="s", color="#1EBFB3", label="Google GRRR")
        ax.set_xlabel("Leadtime (days)")
        ax.set_ylabel(ylabel)
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=9)
        ax.grid(axis="y", linewidth=0.5, alpha=0.5)
        ax.set_title(f"{rp}-year RP — {metric.replace('_', ' ').title()} — {STATE}")

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/{STATE.lower()}_exceedance_by_leadtime.png", dpi=200, bbox_inches="tight")
plt.show()